# Convert Dataset to YOLO26 Format

This notebook converts a custom bacteria dataset into YOLO26 format for object detection training.

## Overview
- **Input Format**: Custom JSON-based annotations with camera images
- **Output Format**: YOLO26 compatible format with normalized bounding box coordinates
- **Output Structure**: Organized train/validation split with images and labels

## Key Features
- Automatically splits dataset into training and validation sets
- Converts custom bounding box format to YOLO normalized coordinates
- Creates class definitions file from annotation definitions
- Generates compressed archive of the final dataset

In [30]:
# Import Required Libraries
import json
import os
import shutil
from PIL import Image

## Utility Functions

The following functions handle the core dataset conversion workflow:

1. **generate_directories()** - Creates the train/validation directory structure
2. **read_annotations()** - Extracts class labels and creates the classes.txt file
3. **create_files()** - Main conversion function that processes images and annotations
4. **zip_directory()** - Creates a compressed archive of the final dataset

### 1. Directory Structure Creation

This function sets up the required directory hierarchy for YOLO format datasets. The structure separates training and validation data, with each split containing subdirectories for images and their corresponding annotation labels.

In [31]:
def generate_directories(experiment_name):
    """
    Create directory structure for YOLO dataset format.
    
    Structure created:
    - experiment_name/train/images/
    - experiment_name/train/labels/
    - experiment_name/valid/images/
    - experiment_name/valid/labels/
    
    Args:
        experiment_name (str): Base path for the experiment directories
    """
    os.makedirs(f"{experiment_name}/train/images", exist_ok=True)
    os.makedirs(f"{experiment_name}/train/labels", exist_ok=True)
    os.makedirs(f"{experiment_name}/valid/images", exist_ok=True)
    os.makedirs(f"{experiment_name}/valid/labels", exist_ok=True)

    print(f"[SUCCESS] Directories created for experiment '{experiment_name}'")

### 2. Class Labels Extraction

This function reads the annotation definitions from the source dataset and creates a `classes.txt` file listing all class labels. This file is essential for YOLO26 training as it maps class IDs to their names.

In [32]:
def read_annotations(image_path, destiny_path):
    """
    Extract class labels from annotation definitions and create classes.txt file.
    
    Reads the annotation_definitions.json file and extracts all class labels,
    then writes them to a classes.txt file compatible with YOLO26 format.
    
    Args:
        image_path (str): Path to source dataset containing annotation_definitions.json
        destiny_path (str): Path where classes.txt will be written
    """
    annotations_file = os.path.join(image_path, 'annotation_definitions.json')
    classes_file = os.path.join(destiny_path, "classes.txt")
    yaml_file = os.path.join(destiny_path, "data.yaml")
    
    print(f"[READ] Reading annotations from: {annotations_file}")
    with open(annotations_file, 'r') as file:
        classes_json_file = json.load(file)

    print(f"[WRITE] Writing class definitions to: {classes_file}")
    with open(classes_file, 'w') as classes_output:
        for classe in classes_json_file['annotationDefinitions'][0]['spec']:
            classes_output.write(f"{classe['label_name']}\n")

    print(f"[WRITE] Writing data.yaml file for YOLO26 format to: {yaml_file}")
    with open(yaml_file, 'w') as yaml_file:
        yaml_file.write(f"train: ../train/images\n")
        yaml_file.write(f"val: ../valid/images\n")
        yaml_file.write(f"nc: {len(classes_json_file['annotationDefinitions'][0]['spec'])}\n")
        yaml_classes = ", ".join([f"'{classe['label_name']}'" for classe in classes_json_file['annotationDefinitions'][0]['spec']])
        yaml_file.write(f"names: [{yaml_classes}]\n")
    
    print(f"[SUCCESS] classes.txt created successfully with {len(classes_json_file['annotationDefinitions'][0]['spec'])} classes")

### 3. Dataset Conversion

This is the main conversion function that:
- Iterates through all sequences in the source dataset
- Splits data into training and validation sets based on the specified ratio
- Copies images to their respective folders
- Converts bounding box annotations from the custom format to YOLO26 format

**Coordinate Conversion**: Each bounding box is converted to the YOLO format where coordinates are normalized (0-1 range) and represented as center coordinates instead of top-left corners:
- `x_center = (left + width/2) / image_width`
- `y_center = (top + height/2) / image_height`
- `width_norm = width / image_width`
- `height_norm = height / image_height`

In [33]:
def create_files(input_path, destiny_path, validation_split=0.2):
    """
    Convert dataset to YOLO26 format with train/validation split.
    
    Processes each sequence in the dataset:
    1. Reads frame data and image dimensions
    2. Splits data into training and validation sets
    3. Copies images to appropriate folders
    4. Converts bounding boxes to YOLO format (normalized center coordinates)
    
    YOLO Format: <class_id> <x_center> <y_center> <width> <height>
    (All coordinates normalized to 0-1 range)
    
    Args:
        input_path (str): Path to source dataset
        destiny_path (str): Path where YOLO26 format files will be saved
        validation_split (float): Fraction of data for validation (default: 0.2)
    """
    print(f"[PROCESS] Processing dataset from: {input_path}")
    print(f"   Destination: {destiny_path}")
    print(f"   Validation split: {validation_split*100:.0f}%\n")

    # Get list of all sequences in the dataset
    dir_list = os.listdir(input_path)
    valid_size = int((len(dir_list) - 4) * validation_split)  # -4 accounts for non-sequence items
    captures_list = [item for item in dir_list if "sequence" in item]
    
    print(f"[INFO] Found {len(captures_list)} sequences")
    print(f"   Validation samples: {valid_size}\n")

    for capture in captures_list:
        # Get image dimensions from first frame
        image_path = os.path.join(input_path, capture, "step0.camera.png")
        frame_data_path = os.path.join(input_path, capture, "step0.frame_data.json")
        
        with Image.open(image_path) as image:
            pic_width, pic_height = image.size
            print(f"[IMAGE] Processing {capture} (Image: {pic_width}x{pic_height})")

        # Read frame data with all captures
        with open(frame_data_path, 'r') as file:
            big_json_file = json.load(file)

        sequence = big_json_file['sequence']
        
        # Process each capture in the sequence
        for i, picture in enumerate(big_json_file['captures']):
            # Determine if image goes to validation or training set
            if valid_size > 0:
                valid_size -= 1
                data_set = "valid"
            else:
                data_set = "train"
            
            # Copy image to appropriate folder
            source_image = os.path.join(input_path, capture, "step0.camera.png")
            dest_image = os.path.join(destiny_path, data_set, "images", 
                                     f"step0.camera1_{sequence}.png")

            try:
                shutil.copy(source_image, dest_image)
            except FileNotFoundError as e:
                print(f"   [WARNING] Error copying image: {e}")
                continue

            # Create annotation file with normalized YOLO coordinates
            filename = picture['filename'].split('/')[-1]
            filename = filename[:-4] + f"{i+1}_{sequence}.txt"
            
            label_path = os.path.join(destiny_path, data_set, "labels", filename)
            with open(label_path, 'w') as annotation_file:
                bbox_count = 0
                for bbox in picture['annotations'][0]['values']:
                    # Convert to YOLO format: normalized center coordinates
                    x_center = (bbox['origin'][0] + bbox['dimension'][0] / 2) / pic_width
                    y_center = (bbox['origin'][1] + bbox['dimension'][1] / 2) / pic_height
                    width = bbox['dimension'][0] / pic_width
                    height = bbox['dimension'][1] / pic_height
                    
                    annotation_file.write(f"{bbox['labelId']} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
                    bbox_count += 1
            
            print(f"   [DONE] {data_set}: {filename} ({bbox_count} objects)")
        
        print()

### 4. Archive Creation

This function compresses the entire converted dataset into a ZIP file for easy distribution and storage.

In [34]:
def zip_directory(source_dir, zip_filename):
    """
    Create a compressed archive of the dataset.
    
    Args:
        source_dir (str): Path to directory to compress
        zip_filename (str): Output filename for the archive (without .zip extension)
    """
    print(f"[ARCHIVE] Creating archive: {zip_filename}.zip")
    shutil.make_archive(zip_filename, 'zip', source_dir)
    print(f"[SUCCESS] Archive created successfully")

## Configuration

In [35]:
# Use the example dataset to test the functions
#import zipfile
#import os

#print("Extracting training dataset...")
#with zipfile.ZipFile("../Resources/Datasets/bacteria-perception-dataset.zip", 'r') as zip_ref:
#    zip_ref.extractall("../Resources/Datasets/bacteria-perception-dataset")
#print("✓ Training dataset extracted")

Edit these parameters to control the conversion process. Modify these values according to your dataset location and desired output directory.

In [36]:
# Configuration Variables
# type of input: folder or file
input_type = "folder"  # Change to "file" if input is a single file
# Path to the source dataset (with custom format)
filename = "chain01-1"

source_dataset = f"..\\Resources\\Experiments\\{filename}"

source_directory = f"..\\Resources\\Experiments"

# Base directory where results will be stored
result_base_path = "..\\Resources\\Results"


# File name
result_filename = filename

# Validation split ratio (0.2 = 20% for validation, 80% for training)
validation_split = 0.2

# Display configuration summary
print("Configuration Settings:")
print(f"  Source Dataset: {source_dataset}")
print(f"  Source Directory: {source_directory}")
print(f"  Output Base Path: {result_base_path}")
print(f"  Input Type: {input_type}")
print(f"  Validation Split: {validation_split*100:.0f}%")

Configuration Settings:
  Source Dataset: ..\Resources\Experiments\chain01-1
  Source Directory: ..\Resources\Experiments
  Output Base Path: ..\Resources\Results
  Input Type: folder
  Validation Split: 20%


## Execute Conversion Pipeline

Run this cell to start the complete conversion workflow. All steps are wrapped in error handling to provide clear feedback if any issues occur.

In [37]:
def execute_pipeline(source_dataset, result_base_path, filename):
    # Create result paths from configuration variables
    result_path = os.path.join(result_base_path, filename)
    zip_file_name = os.path.join(result_base_path, filename)

    print("=" * 60)
    print("YOLO FORMAT CONVERSION PIPELINE")
    print("=" * 60)
    print(f"Source Dataset: {source_dataset}")
    print(f"Output Directory: {result_path}")
    print(f"Archive File: {zip_file_name}.zip")
    print("=" * 60 + "\n")

    # Execute conversion pipeline
    try:
        print("STEP 1: Creating directory structure...")
        generate_directories(result_path)
        print()
        
        print("STEP 2: Extracting class definitions...")
        read_annotations(source_dataset, result_path)
        print()
        
        print("STEP 3: Converting and splitting dataset...")
        create_files(source_dataset, result_path, validation_split=validation_split)
        print()
        
        print("STEP 4: Creating archive...")
        zip_directory(result_path, zip_file_name)
        print()
        
        print("=" * 60)
        print("[SUCCESS] CONVERSION COMPLETED SUCCESSFULLY")
        print("=" * 60)
        
    except Exception as e:
        print(f"\n[ERROR] {e}")
        import traceback
        traceback.print_exc()

In [38]:
import os

if input_type == "folder":
    for filename in os.listdir(source_directory):
        execute_pipeline(f"{source_directory}\\{filename}", result_base_path, filename)
else:
    execute_pipeline(source_dataset, result_base_path, filename)


YOLO FORMAT CONVERSION PIPELINE
Source Dataset: ..\Resources\Experiments\bend10
Output Directory: ..\Resources\Results\bend10
Archive File: ..\Resources\Results\bend10.zip

STEP 1: Creating directory structure...
[SUCCESS] Directories created for experiment '..\Resources\Results\bend10'

STEP 2: Extracting class definitions...
[READ] Reading annotations from: ..\Resources\Experiments\bend10\annotation_definitions.json
[WRITE] Writing class definitions to: ..\Resources\Results\bend10\classes.txt
[WRITE] Writing data.yaml file for YOLO26 format to: ..\Resources\Results\bend10\data.yaml
[SUCCESS] classes.txt created successfully with 3 classes

STEP 3: Converting and splitting dataset...
[PROCESS] Processing dataset from: ..\Resources\Experiments\bend10
   Destination: ..\Resources\Results\bend10
   Validation split: 20%

[INFO] Found 500 sequences
   Validation samples: 100

[IMAGE] Processing sequence.0 (Image: 544x544)
   [DONE] valid: step0.camera1_0.txt (49 objects)

[IMAGE] Processi